<a href="https://colab.research.google.com/github/khouloud-ghabi/Data_Cleaning/blob/main/iris_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning and Preprocessing


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

INPUT_FILE = "1) iris.csv"
OUTPUT_FILE = "iris_cleaned.csv"

## Load the Dataset



In [ ]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded    : {df.shape[0]}")
print(f"Columns loaded : {df.shape[1]}")
print(f"Column names   : {list(df.columns)}")
df.head()

Rows loaded    : 150
Columns loaded : 5
Column names   : ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
df.dtypes

,0
sepal_length,float64
sepal_width,float64
petal_length,float64
petal_width,float64
species,object


## Step 2 — Identify and Handle Missing Values

We count missing values per column, and also compute the percentage — the treatment strategy depends on *how much* is missing, not just whether it is missing.

**Strategy used if missing values are found:**
- Numeric column → impute with the **median** (robust to outliers)
- Categorical column → impute with the **mode** (most frequent value)
- Column with **> 40% missing** → dropped entirely (too sparse to impute reliably)


In [ ]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_%": missing_pct})
missing_report

,missing_count,missing_%
sepal_length,0,0.0
sepal_width,0,0.0
petal_length,0,0.0
petal_width,0,0.0
species,0,0.0


In [ ]:
total_missing = missing_counts.sum()

if total_missing == 0:
    print("No missing values found in any column. No imputation or removal needed.")
else:
    for col in df.columns:
        pct = missing_pct[col]
        if pct == 0:
            continue
        if pct > 40:
            print(f"Dropping column '{col}' ({pct}% missing).")
            df = df.drop(columns=[col])
        elif df[col].dtype.kind in "biufc":
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"Imputed numeric column '{col}' with median = {median_val}")
        else:
            mode_val = df[col].mode(dropna=True)[0]
            df[col] = df[col].fillna(mode_val)
            print(f"Imputed categorical column '{col}' with mode = '{mode_val}'")

No missing values found in any column. No imputation or removal needed.


##  Remove Duplicate Rows


In [ ]:
n_before = len(df)
duplicate_mask = df.duplicated(keep=False)
print(f"Rows involved in duplicate sets: {duplicate_mask.sum()}")
df[duplicate_mask].sort_values(by=list(df.columns))

Rows involved in duplicate sets: 5


,sepal_length,sepal_width,petal_length,petal_width,species
9,4.9,3.1,1.5,0.1,setosa
34,4.9,3.1,1.5,0.1,setosa
37,4.9,3.1,1.5,0.1,setosa
101,5.8,2.7,5.1,1.9,virginica
142,5.8,2.7,5.1,1.9,virginica


In [ ]:
df = df.drop_duplicates(keep="first").reset_index(drop=True)
n_after = len(df)

print(f"Rows before: {n_before}")
print(f"Rows after : {n_after}")
print(f"Rows removed: {n_before - n_after}")

Rows before: 150
Rows after : 147
Rows removed: 3


## Standardize Inconsistent Formats



In [ ]:
old_columns = list(df.columns)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print(f"Column names: {old_columns} -> {list(df.columns)}")

Column names: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'] -> ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


In [ ]:
df["species"] = df["species"].astype(str).str.strip().str.lower()
df["species"].unique()

array(['setosa', 'versicolor', 'virginica'], dtype=object)

In [ ]:
measurement_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
for col in measurement_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.dtypes

,0
sepal_length,float64
sepal_width,float64
petal_length,float64
petal_width,float64
species,object


In [ ]:
invalid = (df[measurement_cols] <= 0).sum().sum()
print(f"Invalid (<= 0) measurement values found: {invalid}")

Invalid (<= 0) measurement values found: 0


In [ ]:
print(f"Final shape             : {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"Remaining duplicate rows: {df.duplicated().sum()}")
print()
print("Class balance ('species'):")
print(df["species"].value_counts())

Final shape             : (147, 5)
Remaining missing values: 0
Remaining duplicate rows: 0

Class balance ('species'):
species
versicolor    50
virginica     49
setosa        48
Name: count, dtype: int64


In [ ]:
df.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,147.000000,147.000000,147.000000,147.000000
mean,5.856463,3.055782,3.780272,1.208844
std,0.829100,0.437009,1.759111,0.757874
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.400000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print(f"Cleaned dataset exported to: {OUTPUT_FILE}")

Cleaned dataset exported to: iris_cleaned.csv
